# Lab 01 · Prompting and small evaluations

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Save `DEEPSEEK_API_KEY` in Colab Secrets and keep `PROVIDER="deepseek"`. Step 4 downloads a small local model; Step 5 uses DeepSeek thinking mode. Other providers work in Steps 1–3; Step 5 requires DeepSeek. Each client stops after 100 calls.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
PROVIDER = "deepseek"  # or "gemini" or "freeinference" for Steps 1–3
from coursekit.runtime import Client, secret
from coursekit.notebook import PROVIDERS
settings = PROVIDERS[PROVIDER]
key = secret(settings["secret_name"])
plain = Client(model=settings["model"], base_url=settings["base_url"],
               api_key=key, max_calls=100, extra_body=settings.get("extra_body"))
thinking = None
if PROVIDER == "deepseek":
    thinking = Client(model=settings["model"], base_url=settings["base_url"],
                      api_key=key, max_calls=100, max_tokens=4096, timeout=120,
                      extra_body={"thinking": {"type": "enabled"}, "reasoning_effort": "low"})

## Warm-up · Separate instructions from the question

Ask “What is a rainbow?” twice with the same instruction to answer as a friendly tutor. The first call puts everything in one message. The second puts the instruction in the system message and the question in the user message. Each call generates a new reply, so compare how the requests are organized even if the answers look similar.

In [ ]:
mixed = plain.ask("You are a friendly tutor. Answer in one sentence. What is a rainbow?", system="")
print("Mixed:", mixed)

split = plain.ask("What is a rainbow?", system="You are a friendly tutor. Answer in one sentence.")
print("Separated:", split)

**Next:** Reuse a tutor instruction on two architecture questions, then carry an answer forward and check a reply.

## Prompts, memory and evaluation
In Lab 00 you added examples or a role to a prompt. Here you will put reusable instructions in their own message, give the model the context it needs for a follow-up, and write a check for its answers.

ShopEasy has web and mobile apps. An **API gateway** gives clients a shared point of entry. A **backend for frontend (BFF)** can give a particular client the data and operations it needs. We use these two options to make the model’s answers easy to check.

Start with Steps 1–3. The sampling interlude explains how the model picks text. Steps 4 and 5 compare ways to answer two short puzzles whose answers you can check with Python.

**At a glance**

![Lab 01 at a glance: Architecture question (gateway or BFF), then Prompt (system + user), then Remember (pass prior answer), then Validate (JSON + decision), then Compare (older + thinking modes).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/01.svg)

## Think about the example

An API gateway can route requests and apply common policies. A backend for frontend (BFF) can tailor an API to one client, such as a mobile app. A system may use both.

**Concepts:** API gateway; Backend for frontend (BFF); Client-specific APIs.

Use the scenario above to consider these questions before running the code:

1. What shared needs can an API gateway address?
2. Why might mobile and web clients benefit from different BFFs?

## The idea in context

A system message guides the assistant; a user message requests a task. A follow-up can include a previous answer as temporary context.

Parse and validate replies before using them. An older model and DeepSeek thinking modes can be compared on small, checkable examples.

| Key term | Meaning |
| --- | --- |
| **System prompt** | Instructions that guide the assistant’s task and response. |
| **User prompt** | The specific question or request sent by the user. |
| **Temporary context** | Information, such as a previous answer, sent again in a later request. |
| **Validation** | Checking that a response has the expected format and meets a specific condition. |
| **Thinking mode** | A model setting that enables additional internal reasoning before the final answer. |

## Main exercise at a glance

1. From one mixed prompt to two roles
2. Temporary memory: pass the previous answer
3. Validate a structured answer, three times
4. See a small model make mistakes
5. Compare DeepSeek without and with thinking

## Step 1 · From one mixed prompt to two roles
A request can contain messages with different roles. The **system message** gives the model reusable instructions, such as how to behave and how long its answers should be. The **user message** gives it the current question and facts.

The first call puts the instructions and question together in one user message. `system=""` leaves the system message empty. The second sends the same wording as two messages. The third keeps the tutor instructions and changes the question. Look for the tailored mobile API in the first two replies and the shared routing entry point in the third.

For a larger system, some providers can [reuse the processing of repeated prompt text](https://api-docs.deepseek.com/guides/kv_cache/). This example focuses on organizing the messages.

In [ ]:
mixed_prompt = """You are a software architecture tutor. Answer in one sentence.
ShopEasy needs a tailored API for its mobile app. What pattern fits?"""
mixed_answer = plain.ask(mixed_prompt, system="")

system_prompt = "You are a software architecture tutor. Answer in one sentence."
user_prompt = "ShopEasy needs a tailored API for its mobile app. What pattern fits?"
split_answer = plain.ask(user_prompt, system=system_prompt)

new_question = "For several services, which provides one routing entry point: an API gateway or a BFF?"
next_answer = plain.ask(new_question, system=system_prompt)
print("Mixed prompt:", mixed_answer)
print("Split prompt:", split_answer)
print("Same system prompt, new question:", next_answer)

## Step 2 · Temporary memory: pass the previous answer
The first request asks “What is one drawback of this choice?” on its own. The model has to guess what “this choice” refers to or ask for more context.

Next, ask for an architecture recommendation and save the returned text in `first_answer`. Put that text into the follow-up prompt and send it again. Read `Without context`, `First answer`, and `Follow-up` in the output. Which follow-up can refer to a specific design?

![Temporary memory: without context, a follow-up does not identify the choice. With context, save the first model answer in first_answer and include it in the second request along with the follow-up question.](https://mkassaf.github.io/agentic-ai-course/assets/images/temporary-memory.svg)

In these direct API calls, the Python variable holds the context between requests. A chat app can send earlier turns as separate user and assistant messages.

In [ ]:
without_context = plain.ask("What is one drawback of this choice? Answer in one sentence.")

first_answer = plain.ask(
    "ShopEasy's mobile app needs a tailored API. Suggest one pattern in one sentence.")

follow_up = f"""Earlier answer: {first_answer}
What is one drawback of this choice? Answer in one sentence."""
second_answer = plain.ask(follow_up)
print("Without context:", without_context)
print("First answer:", first_answer)
print("Follow-up:", second_answer)

## Step 3 · Validate a structured answer, three times
Ask the same architecture question three ways and request **JSON**, a text format Python can read. One expected reply looks like `{"decision": "BFF", "reason": "The mobile app needs its own API."}`.

The `check(reply)` function returns two results. **Valid format** means Python could read the reply and find an allowed decision and a written reason. **Expected decision** means the reply also chose BFF. Read the reason yourself to see whether it fits the case.

The cell prints both checks for each of three requests, then a pass count. This is a small **evaluation**, where you compare model outputs to checks you define in advance.

In [ ]:
import json

def check(reply):
    """Return (valid_format, expected_decision) for one architecture reply."""
    try:
        answer = json.loads(reply)
    except json.JSONDecodeError:
        return False, False
    valid = (isinstance(answer, dict) and answer.get("decision") in ["BFF", "API gateway"]
             and isinstance(answer.get("reason"), str) and bool(answer["reason"].strip()))
    return valid, valid and answer["decision"] == "BFF"

INSTRUCTIONS = ('Choose BFF or API gateway. Return JSON only, with keys "decision" and "reason".'
                " Give a short reason.")
prompts = [
    f"Web and mobile clients need different tailored APIs. Which pattern fits? {INSTRUCTIONS}",
    f"The mobile app needs a compact, dedicated API while the web site keeps rich pages. "
    f"What should the mobile team ask for? {INSTRUCTIONS}",
    f"One shared entry point routes all clients today; now the web UI wants its own tailored API too. "
    f"Add a BFF or keep only the API gateway? {INSTRUCTIONS}"]
passed = 0
for prompt in prompts:
    reply = plain.ask(prompt)
    valid, correct = check(reply)
    passed += correct
    print("Valid format:", valid, "| Expected decision:", correct, "| Reply:", reply)
print(f"Small evaluation: {passed}/{len(prompts)} replies matched the expected decision.")

## Sampling interlude · How temperature, top-k, and top-p work together

A **token** is a unit of text, such as a word or part of a word. A model generates a reply one token at a time. At each step, it scores possible next tokens. This illustration shows one way to turn those scores into a choice.

1. **Temperature** changes the odds. Higher values spread probability more evenly across the candidates.
2. **Top-k** keeps up to a fixed number of the highest-ranked candidates.
3. **Top-p** keeps the most likely candidates whose combined probability reaches a threshold.

![Sampling example: four scores become probabilities after temperature 2. Top-k 3 removes D; top-p 0.80 keeps A and B; the model then samples A with 62.25% or B with 37.75%.](https://mkassaf.github.io/agentic-ai-course/assets/images/sampling-settings.svg)

In this worked example, top-k keeps three choices, then top-p keeps two. The model picks from those two using the updated odds. This order follows [Transformers for these settings](https://github.com/huggingface/transformers/blob/main/src/transformers/generation/utils.py) when sampling without other filters. Model services can apply these settings differently.

## Sampling interlude · Optional: explore the simulator

Move the temperature, top-k, and top-p sliders to see which candidate tokens remain and how their probabilities change. Use **Post-Temp** to inspect the probabilities before the filters, and **Final Odds** to see the distribution after them. Try changing one setting at a time.

The simulator uses illustrative probabilities; it does not call an LLM or need a key. It applies temperature → top-k → top-p, then normalizes the remaining choices. Providers can use different rules. The code below embeds it in Colab; if the frame does not load, [open the simulator in a new tab](https://mkassaf.github.io/agentic-ai-course/assets/tools/sampling.html).

In [ ]:
from IPython.display import IFrame, display

display(IFrame("https://mkassaf.github.io/agentic-ai-course/assets/tools/sampling.html",
               width="100%", height=1100))

## Sampling interlude · Optional: change the temperature

A model chooses its next token from possible continuations. **Temperature** changes how strongly it favors higher-probability choices: a lower value generally makes the distribution more concentrated; a higher value makes it flatter. It does not directly control factual accuracy, and a low value does not guarantee identical answers.

Run setup and Step 1 first. Keep the same architecture question, system instruction, model, and output limit. Make one live request at `temperature=0.0` and another at `1.3`. Compare the answers, then rerun to see whether they change. Two answers may be identical; two samples cannot establish which setting is better.

`extra_body` supplies the model's API parameters. The existing provider options are preserved, including DeepSeek's disabled-thinking setting. Each client has its own one-call limit; these are two additional model requests. Use a value supported by your selected provider. [DeepSeek documents temperature from 0 to 2](https://api-docs.deepseek.com/api/create-chat-completion/) and says it has no effect in thinking mode; this example uses thinking disabled.

In [ ]:
config_question = "ShopEasy has web and mobile clients. In one sentence, suggest one trade-off of adding a BFF."
low_options = {**(settings.get("extra_body") or {}), "temperature": 0.0}
high_options = {**(settings.get("extra_body") or {}), "temperature": 1.3}

low = Client(model=settings["model"], base_url=settings["base_url"], api_key=key,
             max_calls=1, max_tokens=128, extra_body=low_options)
high = Client(model=settings["model"], base_url=settings["base_url"], api_key=key,
              max_calls=1, max_tokens=128, extra_body=high_options)
print("Temperature 0.0:", low.ask(config_question, system=system_prompt))
print("Temperature 1.3:", high.ask(config_question, system=system_prompt))

## Step 4 · See a small model make mistakes
Run [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct), a small model, on two questions. For each question, first ask directly, then add concrete solving steps to the prompt. This is one form of **chain-of-thought prompting**, where you ask the model to work through intermediate steps.

Compare both replies with the Python checks printed underneath. The word strawberry contains **3** r's, and **9.9** is larger than 9.11. In one run both direct answers were wrong. The added steps helped the model choose 9.9, but it repeated letters without giving a final count for strawberry. Your replies can differ. Check both the final answer and the steps it describes.

The first run downloads the model and can take several minutes. It runs on the Colab CPU. You can continue to Step 5 if this download is unavailable.

In [ ]:
if find_spec("transformers") is None:
    %pip install -q "transformers>=4.45,<5" torch
from transformers import pipeline
from decimal import Decimal

old = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct", device=-1)
def ask_old(prompt):
    result = old([{"role": "user", "content": prompt}], max_new_tokens=96, do_sample=False)
    return result[0]["generated_text"][-1]["content"]

letters = "How many times does the letter 'r' appear in strawberry? Reply with only a number."
decimals = ("Compare the decimal numbers 9.9 and 9.11. "
            "Which is greater? Reply with only the greater decimal.")
guided_letters = ("How many r's are in strawberry? Think step by step, "
                  "list the letters in the word, and count the r's. Finish with the number.")
guided_decimals = ("Which decimal is larger, 9.9 or 9.11? Step 1: write both with two digits "
                   "after the decimal point. Step 2: compare the values. Then answer.")
print("Strawberry — direct:", ask_old(letters))
print("Strawberry — guided:", ask_old(guided_letters))
print("Correct count (Python):", "strawberry".count("r"))
print("Decimals — direct:", ask_old(decimals))
print("Decimals — guided:", ask_old(guided_decimals))
print("Correct larger decimal (Python):", max(Decimal("9.9"), Decimal("9.11")))

## Step 4a · Optional: change top-k

**Top-k** limits the next-token choices to the `k` highest-probability candidates. With `top_k=1`, the local model selects its highest-probability token at each step; with `top_k=20`, it may sample among a wider set. **Top-p** instead keeps a variable-sized set whose cumulative probability reaches a chosen threshold. `max_new_tokens` limits response length and may cut an answer off.

The picture in the sampling interlude above walks through a numerical example of the two filters.

This example uses the local SmolLM model loaded in Step 4, not DeepSeek. Hugging Face sampling requires `do_sample=True`; we keep `temperature=0.8` and `top_p=1.0` the same and change only `top_k`. Run Step 4 first. Resetting the random seed makes the comparison easier to repeat in the same environment; outputs can still match or be wrong, and results can vary across environments. This tiny model may produce irrelevant or incorrect architecture advice; inspect its answers. [Hugging Face generation settings](https://huggingface.co/docs/transformers/main_classes/text_generation)

DeepSeek's [Chat Completions API](https://api-docs.deepseek.com/api/create-chat-completion/) documents `temperature` and `top_p`, but not `top_k`. For the default `deepseek-flash` model, `top_p` is fixed at 1.0 when thinking is disabled, so changing `top_p` has no effect in that mode. Other providers and API endpoints can differ; check their documentation before sending a parameter.

In [ ]:
from transformers import set_seed

top_k_question = "Choose a way to handle a slow Inventory service: retry, timeout, or circuit breaker. Answer with one choice."
for k in (1, 20):
    set_seed(42)
    result = old([{"role": "user", "content": top_k_question}], max_new_tokens=40,
                 do_sample=True, temperature=0.8, top_k=k, top_p=1.0)
    print("Top-k", k, ":", result[0]["generated_text"][-1]["content"])

## Step 5 · Compare DeepSeek without and with thinking
Send the same two puzzle questions to the same DeepSeek model in two modes. With **thinking enabled**, the model can spend more computation on intermediate reasoning before producing its final reply. You will see the final reply for each mode and the Python checks.

The prompt stays fixed in this comparison. The thinking client also has a larger output-token limit and a longer timeout so the model has room to reason. Compare the replies with the check outputs. Either mode can make mistakes.

This step uses a `DEEPSEEK_API_KEY` in Colab Secrets with `PROVIDER = "deepseek"` in setup. If you have a different provider key, the cell prints only the Python checks and the instruction for switching. This step can run independently of the local model from Step 4.

In [ ]:
from decimal import Decimal

letters = "How many times does the letter 'r' appear in strawberry? Reply with only a number."
decimals = "Compare the decimal numbers 9.9 and 9.11. Which is greater? Reply with only the greater decimal."

if thinking is None:
    print("Switch PROVIDER to deepseek in setup to run the thinking comparison.")
else:
    print("DeepSeek, no thinking:", plain.ask(letters))
    print("DeepSeek, thinking:", thinking.ask(letters))
    print("DeepSeek, no thinking:", plain.ask(decimals))
    print("DeepSeek, thinking:", thinking.ask(decimals))
print("Python checks: r count =", "strawberry".count("r"))
print("Python checks: 9.9 > 9.11 is", Decimal("9.9") > Decimal("9.11"))

## What to look for
- **Steps 1–2:** Find the reused tutor instruction and the previous answer sent with the follow-up.
- **Step 3:** Compare the displayed format and decision checks, then read the reason yourself.
- **Steps 4–5:** Use the Python answers to check each model reply. A correct answer may still come with an incorrect explanation.

In the optional sampling examples, note whether the wording changes as you vary one setting.

## Try it
1. In Step 1, keep the system prompt the same and write another architecture question. Which parts can be reused?
2. Remove the earlier answer from `follow_up` in Step 2. What context is missing?
3. Call `check()` with a JSON string whose decision is `API gateway`, then with text that is not JSON at all. What does each result show?
4. Change one sampling setting at a time in the optional examples. Do repeated responses always change?
5. Report the actual answers from both DeepSeek modes. Did the guided older-model prompt help? Repeat if you have time; these two examples are not a benchmark.

## Takeaway
A system message sets behavior, a user message asks for a task, and temporary context must be sent again. Validate outputs and test reasoning claims rather than assuming a model or prompt is always right.